In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd
import os

# ── Load employee data ──────────────────────────────────────────────────────
CSV_FILE = "GIG-logistics.csv"

def load_data() -> pd.DataFrame:
    """Load and clean the GIG Logistics employee CSV."""
    if not os.path.exists(CSV_FILE):
        messagebox.showerror(
            "File Not Found",
            f"'{CSV_FILE}' was not found.\n"
            "Please place it in the same folder as this script."
        )
        raise SystemExit
    df = pd.read_csv(CSV_FILE)
    # Strip whitespace from all string columns
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()
    return df

df = load_data()

# Column references
SN_COL    = "S/N"
SUR_COL   = "SURNAME"
FIRST_COL = "FIRST NAME"
DEPT_COL  = "DEPARTMENT"


# ── Lookup logic ─────────────────────────────────────────────────────────────
def lookup_employee(surname: str, firstname: str, department: str):
    """
    Returns (found: bool, row: Series | None, colleagues: DataFrame).
    Matching is case-insensitive.
    """
    sur   = surname.strip().upper()
    first = firstname.strip().title()
    dept  = department.strip()

    mask = (
        df[SUR_COL].str.upper()   == sur
    ) & (
        df[FIRST_COL].str.title() == first
    ) & (
        df[DEPT_COL].str.lower()  == dept.lower()
    )
    match = df[mask]

    if match.empty:
        return False, None, pd.DataFrame()

    row = match.iloc[0]
    colleagues = df[
        (df[DEPT_COL].str.lower() == dept.lower()) &
        (df.index != row.name)          # exclude the matched employee
    ]
    return True, row, colleagues


# ── GUI ───────────────────────────────────────────────────────────────────────
class App(tk.Tk):
    NAVY   = "#0D1B3E"
    GOLD   = "#F5C518"
    WHITE  = "#FFFFFF"
    LIGHT  = "#E8EAF0"
    GREEN  = "#2ECC71"
    RED    = "#E74C3C"
    SILVER = "#B0B8C8"

    def __init__(self):
        super().__init__()
        self.title("GIG Logistics – Employee Verification")
        self.resizable(False, False)
        self.configure(bg=self.NAVY)
        self._build_styles()
        self._build_ui()
        self._center_window(720, 620)

    def _center_window(self, w: int, h: int):
        self.update_idletasks()
        x = (self.winfo_screenwidth()  - w) // 2
        y = (self.winfo_screenheight() - h) // 2
        self.geometry(f"{w}x{h}+{x}+{y}")

    def _build_styles(self):
        style = ttk.Style()
        style.theme_use("clam")
        style.configure("GIG.Treeview",
                        background=self.LIGHT, foreground=self.NAVY,
                        rowheight=26, font=("Arial", 10),
                        fieldbackground=self.LIGHT)
        style.configure("GIG.Treeview.Heading",
                        background=self.NAVY, foreground=self.GOLD,
                        font=("Arial", 10, "bold"), relief="flat")
        style.map("GIG.Treeview",
                  background=[("selected", "#c0c8e0")])

    def _build_ui(self):
        # ── Header ────────────────────────────────────────────────────────────
        hdr = tk.Frame(self, bg=self.NAVY, pady=16)
        hdr.pack(fill="x")
        tk.Label(hdr, text="GIG LOGISTICS",
                 font=("Arial", 24, "bold"),
                 bg=self.NAVY, fg=self.GOLD).pack()
        tk.Label(hdr, text="Employee Verification Portal",
                 font=("Arial", 11),
                 bg=self.NAVY, fg=self.SILVER).pack()
        tk.Frame(self, bg=self.GOLD, height=3).pack(fill="x")

        # ── Input card ────────────────────────────────────────────────────────
        card = tk.Frame(self, bg=self.WHITE, padx=28, pady=22)
        card.pack(fill="x", padx=28, pady=(18, 0))

        for col_idx, label in enumerate(["Surname", "First Name", "Department"]):
            tk.Label(card, text=label, font=("Arial", 10, "bold"),
                     bg=self.WHITE, fg=self.NAVY, anchor="w").grid(
                     row=0, column=col_idx, sticky="w",
                     padx=(0, 12), pady=(0, 4))

        # Surname entry
        self.sur_var = tk.StringVar()
        sur_entry = tk.Entry(card, textvariable=self.sur_var,
                             font=("Arial", 12), relief="solid", bd=1, width=20)
        sur_entry.grid(row=1, column=0, padx=(0, 12), ipady=6)
        sur_entry.bind("<Return>", lambda e: self._verify())

        # First name entry
        self.first_var = tk.StringVar()
        first_entry = tk.Entry(card, textvariable=self.first_var,
                               font=("Arial", 12), relief="solid", bd=1, width=20)
        first_entry.grid(row=1, column=1, padx=(0, 12), ipady=6)
        first_entry.bind("<Return>", lambda e: self._verify())

        # Department dropdown – populated from real CSV
        departments = sorted(df[DEPT_COL].dropna().unique().tolist())
        self.dept_var = tk.StringVar()
        dept_combo = ttk.Combobox(card, textvariable=self.dept_var,
                                  values=departments, font=("Arial", 12),
                                  width=17, state="readonly")
        dept_combo.grid(row=1, column=2, ipady=6)
        if departments:
            dept_combo.current(0)

        # ── Buttons ───────────────────────────────────────────────────────────
        btn_frame = tk.Frame(self, bg=self.NAVY, pady=14)
        btn_frame.pack()

        tk.Button(btn_frame, text="  ✔  Verify Employee  ",
                  font=("Arial", 12, "bold"),
                  bg=self.GOLD, fg=self.NAVY,
                  relief="flat", cursor="hand2",
                  activebackground="#d4a800",
                  padx=10, pady=6,
                  command=self._verify).pack(side="left", padx=6)

        tk.Button(btn_frame, text="  ✖  Clear  ",
                  font=("Arial", 12),
                  bg="#3a4a6b", fg=self.WHITE,
                  relief="flat", cursor="hand2",
                  activebackground="#4a5a7b",
                  padx=10, pady=6,
                  command=self._clear).pack(side="left", padx=6)

        # ── Result banner ─────────────────────────────────────────────────────
        res_frame = tk.Frame(self, bg=self.NAVY)
        res_frame.pack(fill="x", padx=28)
        self.result_label = tk.Label(res_frame, text="",
                                     font=("Arial", 13, "bold"),
                                     bg=self.NAVY, fg=self.WHITE,
                                     wraplength=660, justify="center")
        self.result_label.pack(pady=8)

        # ── Colleagues table ──────────────────────────────────────────────────
        tbl_outer = tk.Frame(self, bg=self.NAVY, padx=28)
        tbl_outer.pack(fill="both", expand=True, pady=(0, 20))

        self.tbl_title = tk.Label(tbl_outer, text="",
                                  font=("Arial", 11, "bold"),
                                  bg=self.NAVY, fg=self.GOLD, anchor="w")
        self.tbl_title.pack(fill="x", pady=(0, 6))

        tree_wrap = tk.Frame(tbl_outer, bg=self.NAVY)
        tree_wrap.pack(fill="both", expand=True)

        cols = [SN_COL, SUR_COL, FIRST_COL, DEPT_COL]
        col_widths = {SN_COL: 50, SUR_COL: 200, FIRST_COL: 190, DEPT_COL: 150}

        self.tree = ttk.Treeview(tree_wrap, columns=cols, show="headings",
                                 style="GIG.Treeview", height=7)
        for col in cols:
            self.tree.heading(col, text=col)
            self.tree.column(col, anchor="center", width=col_widths[col])

        vsb = ttk.Scrollbar(tree_wrap, orient="vertical", command=self.tree.yview)
        self.tree.configure(yscrollcommand=vsb.set)
        self.tree.pack(side="left", fill="both", expand=True)
        vsb.pack(side="right", fill="y")

        self.tree.tag_configure("odd",  background="#dde2ef")
        self.tree.tag_configure("even", background=self.LIGHT)

    # ── Actions ───────────────────────────────────────────────────────────────
    def _verify(self):
        surname   = self.sur_var.get().strip()
        firstname = self.first_var.get().strip()
        dept      = self.dept_var.get().strip()

        if not surname or not firstname:
            messagebox.showwarning("Input Required",
                                   "Please enter both Surname and First Name.")
            return

        found, row, colleagues = lookup_employee(surname, firstname, dept)
        self._clear_table()

        if found:
            self.result_label.config(
                text=f"✔  Welcome, {row[FIRST_COL]} {row[SUR_COL]}!  "
                     f"You are verified as a member of the {row[DEPT_COL]} department.",
                fg=self.GREEN)
            if colleagues.empty:
                self.tbl_title.config(
                    text=f"No other employees found in the {dept} department.")
            else:
                self.tbl_title.config(
                    text=f"Other members of the {dept} department "
                         f"({len(colleagues)} employee(s)):")
                for i, (_, r) in enumerate(colleagues.iterrows()):
                    tag = "odd" if i % 2 else "even"
                    self.tree.insert("", "end", tags=(tag,),
                                     values=(r[SN_COL], r[SUR_COL],
                                             r[FIRST_COL], r[DEPT_COL]))
        else:
            full = f"{surname.upper()} {firstname.title()}"
            self.result_label.config(
                text=f"✘  Sorry, '{full}' does not exist as an employee "
                     f"in the {dept} department.",
                fg=self.RED)
            self.tbl_title.config(text="")

    def _clear_table(self):
        for item in self.tree.get_children():
            self.tree.delete(item)

    def _clear(self):
        self.sur_var.set("")
        self.first_var.set("")
        self.result_label.config(text="")
        self.tbl_title.config(text="")
        self._clear_table()


# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    app = App()
    app.mainloop()

C:\Users\joanu\AppData\Local\Temp\ipykernel_7384\3186070116.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


KeyboardInterrupt: 

: 